

# From scratch implementation of FFNN

## Table of Contents
- [1. Imports](#1-imports)
- [2. Activations](#2-activations)
- [3. Initializers](#3-initializers)
- [4. Layers](#4-layers)
- [5. Losses](#5-losses)
- [6. Optimizers](#6-optimizers)
- [7. Neural Network](#7-neural-network)
- [8. Training & Evaluation](#8-training--evaluation)

---


## 1. Imports



In [ ]:
import numpy as np
from .initializers import initialize_weights
from .activations import get_activation, get_activation_derivative
from .layers import DenseLayer
from .losses import get_loss_function, get_loss_derivative, l2_regularization
from .optimizers import get_optimizer

## 2. Activations

Below implements activation functions and their derivatives. The implementation defines the activation functions used in the neural network, together with their corresponding derivatives required for backpropagation. ReLU, sigmoid, and tanh are used for hidden layers, while softmax is applied in the output layer for multi-class classification.

To let the estimates represent class propability in multi-class (single label) classification, the softmax activation function is applied in the output layer.

The predicted probability for class $c$ is:

$$
\hat{y}_c = p(y = c) = \frac{\exp(b_c + w_c^\top h^{(L)})}{\sum_{k=1}^{K} \exp(b_k + w_k^\top h^{(L)})}
$$

The predicted class is then chosen as:

$$
c_{\text{pred}} = \arg\max_c \hat{y}_c
$$


The softmax ensures the outputs are valid probabilities (non-negative and summing to one). The classifier assigns the image to the class with the highest probability.




In [ ]:
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def sigmoid(x):
    x_clipped = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x_clipped))

def sigmoid_derivative(x):
    return sigmoid(x) * (1 - sigmoid(x))

def tanh(x):
    return np.tanh(x)

def tanh_derivative(x):
    return 1 - np.tanh(x)**2

def softmax(x):
    x_shifted = x - np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def softmax_derivative(x):
    return softmax(x) - np.square(softmax(x))

ACTIVATION_FUNCTIONS = {
    'relu': relu,
    'sigmoid': sigmoid,
    'tanh': tanh,
    'softmax': softmax
}

ACTIVATION_DERIVATIVES = {
    'relu': relu_derivative,
    'sigmoid': sigmoid_derivative,
    'tanh': tanh_derivative,
    'softmax': softmax_derivative
}

def get_activation(name):
    if name not in ACTIVATION_FUNCTIONS:
        raise ValueError(f"Unknown activation function: {name}")
    return ACTIVATION_FUNCTIONS[name]

def get_activation_derivative(name):
    if name not in ACTIVATION_DERIVATIVES:
        raise ValueError(f"Unknown activation function: {name}")
    return ACTIVATION_DERIVATIVES[name]


## 3. Initializers
Weight and bias initializers are implemented. 
- Random initialization provides small uniform values as a simple baseline. 
- Xavier (Glorot) initialization scales weights according to the layer’s fan-in and fan-out, making it suitable for sigmoid and tanh activations. 
- He initialization adjusts variance to support ReLU activations by preventing vanishing or exploding gradients. 
- Biases are initialized to zero.




In [ ]:
def random_initialization(shape, seed=None):
    if seed is not None:
        np.random.seed(seed)
    return np.random.uniform(-0.01, 0.01, size=shape)

def xavier_initialization(shape, alpha=1.0, seed=None):
    if len(shape) < 2:
        raise ValueError("Xavier initialization requires at least 2D shape")
    n_in = shape[0]
    n_out = shape[1] if len(shape) > 1 else 1
    limit = np.sqrt(6.0 / (n_in + n_out))
    if seed is not None:
        np.random.seed(seed)
    return np.random.uniform(-limit, limit, size=shape)

def he_initialization(shape, alpha=2.0, seed=None):
    if len(shape) < 2:
        raise ValueError("He initialization requires at least 2D shape")
    n_in = shape[0]
    std = np.sqrt(2.0 / n_in)
    if seed is not None:
        np.random.seed(seed)
    return np.random.normal(0.0, std, size=shape)

def zeros_initialization(shape):
    return np.zeros(shape)

def get_alpha_from_activation(activation):
    activation = activation.lower()
    if activation in ["tanh", "sigmoid"]:
        return 1.0
    elif activation in ["relu"]:
        return 2.0
    else:
        return 1.0

INITIALIZERS = {
    'random': random_initialization,
    'xavier': xavier_initialization,
    'he': he_initialization,
    'zeros': zeros_initialization
}

def get_initializer(name):
    if name not in INITIALIZERS:
        raise ValueError(f"Unknown initializer: {name}")
    return INITIALIZERS[name]

def initialize_weights(input_size, output_size, method='xavier', activation='relu', seed=None):
    initializer = get_initializer(method)
    weight_shape = (input_size, output_size)
    if method == 'zeros':
        weights = initializer(weight_shape)
    else:
        weights = initializer(weight_shape, seed=seed)
    bias_shape = (output_size,)
    biases = zeros_initialization(bias_shape)
    return weights, biases


## 4. Layers

Below block defines a fully connected (dense) layer class. Each layer initializes its weight matrix and bias vector using the selected initialization method, and stores the the chosen activation function. During forward propagation, the layer caches intermediate values—inputs, pre-activations, and activations, storing them for backpropagation. The class also allocates space for gradients (dW and db), which are computed during the backward pass and used to update the parameters.




In [ ]:
class DenseLayer:
    def __init__(self, input_size, output_size, activation='relu',
                 weight_init='xavier', seed=None):
        self.W, self.b = initialize_weights(
            input_size,
            output_size,
            method=weight_init,
            seed=seed
        )
        self.activation = activation
        self.activation_cache = {
            'A_prev': None,
            'Z': None,
            'A': None
        }
        self.dW = None
        self.db = None


## 5. Losses

Below defines the loss functions used to train the network; mean squared error for regression and cross-entropy for classification. It also includes functions to calculate the gradient of each loss for backpropagation, clips values to avoid errors, and supports batches. L2 regularization and its gradient are included too. The module lets the network choose a loss function and its derivative by name.

#### Ridge Regularization (L2)

A common approach is weight decay (L2 regularization), which penalizes large weights by adding a quadratic term to the loss and thereby encourages smoother functions. 

$$
\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{data}} + \lambda \sum_j \theta_j^2
$$

where:
- $\mathcal{L}_{\text{data}}$ is the original loss function,  
- $\theta_j$ are the model parameters, and  
- $\lambda$ controls the strength of regularization.  

Smaller $\lambda$ reduces the effect, while larger $\lambda$ enforces stronger weight shrinkage. Different layers can use different $\lambda$ values if desired.  

#### MNIST and CIFAR-10
For MNIST and CIFAR-10, the target is categorical, so training minimizes the **cross-entropy loss**. The cross-entropy loss used for training is:

$$
\mathcal{L} = -\frac{1}{N} \sum_{j=1}^{N} \sum_{n=1}^{J} y_{j,c} \log(\hat{y}_{j,c})
$$

where $y_{j,c}$ is 1 if sample $j$ belongs to class $c$, and 0 otherwise.

Cross-entropy ensures the network outputs valid class probabilities and learns to assign the highest probability to the correct class.




In [ ]:
# Loss functons for training

def min_log_likelihood(y_pred, y_true):
    eps = 1e-12
    y_pred_clipped = np.clip(y_pred, eps, 1.0 - eps)
    if y_pred_clipped.ndim == 1:
        y_pred_clipped = y_pred_clipped.reshape(1, -1)
        y_true = y_true.reshape(1, -1)
    MLL = -np.sum(y_true * np.log(y_pred_clipped))
    return float(MLL)

def mean_squared_error(y_pred, y_true):
    MSE = np.mean(np.square(y_pred - y_true))
    return float(MSE)

def mse_derivative(y_pred, y_true):
    n = y_pred.size
    dMSE = (2/n) * (y_pred - y_true)
    return dMSE

def cross_entropy_loss(y_pred, y_true):
    eps = 1e-12
    y_pred_clipped = np.clip(y_pred, eps, 1.0 - eps)
    if y_pred_clipped.ndim == 1:
        y_pred_clipped = y_pred_clipped.reshape(1, -1)
        y_true = y_true.reshape(1, -1)
    loss = -np.sum(y_true * np.log(y_pred_clipped)) / y_pred_clipped.shape[0]
    return float(loss)

def cross_entropy_derivative(y_pred, y_true):
    if y_pred.ndim == 1:
        y_pred = y_pred.reshape(1, -1)
        y_true = y_true.reshape(1, -1)
    n = y_pred.shape[0]
    return (y_pred - y_true) / n

def binary_cross_entropy(y_pred, y_true):
    eps = 1e-12
    y_pred_clipped = np.clip(y_pred, eps, 1.0 - eps)
    loss = -np.mean(y_true * np.log(y_pred_clipped) + (1 - y_true) * np.log(1 - y_pred_clipped))
    return float(loss)

def l2_regularization(weights, lambda_):
    l2_sum = 0.0
    for W in weights:
        l2_sum += np.sum(np.square(W))
    L2 = (lambda_ / 2) * l2_sum
    return L2

def l2_regularization_derivative(weight, lambda_):
    return lambda_ * weight

LOSS_FUNCTIONS = {
    'mse': mean_squared_error,
    'cross_entropy': cross_entropy_loss,
    'binary_cross_entropy': binary_cross_entropy
}

LOSS_DERIVATIVES = {
    'mse': mse_derivative,
    'cross_entropy': cross_entropy_derivative,
    'binary_cross_entropy': cross_entropy_derivative
}

def get_loss_function(name):
    if name not in LOSS_FUNCTIONS:
        raise ValueError(f"Unknown loss function: {name}")
    return LOSS_FUNCTIONS[name]

def get_loss_derivative(name):
    if name not in LOSS_DERIVATIVES:
        raise ValueError(f"Unknown loss function: {name}")
    return LOSS_DERIVATIVES[name]


## 6. Optimizers
Following implements optimizer options: SGD, Momentum, RMSProp, and Adam.

- SGD: basic gradient descent moving weights “downhill” along the negative gradient
- MomentumSGD: adds momentum to speed up convergence and smooth updates
- RMSprop: adapts the learning rate for aech parameter based on recent gradient magnitudes
- Adam: combines momentum and adaptive learning rates for ffaster and more stable training



In [ ]:

class Optimizer:
    def __init__(self, learning_rate=0.01):
        self.learning_rate = learning_rate
    
    def update(self, params, grads):
        raise NotImplementedError

class SGD(Optimizer):
    # SGD: W = W - lr * grad
    def __init__(self, learning_rate=0.01):
        super().__init__(learning_rate)
    
    def update(self, params, grads):
        updated_params = {}
        for key in params:
            if key not in grads:
                raise ValueError(f"Gradient for parameter '{key}' not found")
            updated_params[key] = params[key] - self.learning_rate * grads[key]
        return updated_params

class MomentumSGD(Optimizer):
    # SGD with momentum: v = beta*v - lr*grad, W = W + v
    def __init__(self, learning_rate=0.01, momentum=0.9):
        super().__init__(learning_rate)
        self.momentum = momentum
        self.velocity = {}  # Velocity for each param
    
    def update(self, params, grads):
        updated_params = {}
        for key in params:
            if key not in grads:
                raise ValueError(f"Gradient for parameter '{key}' not found")
            if key not in self.velocity:
                self.velocity[key] = np.zeros_like(params[key])
            # Update velocity
            self.velocity[key] = self.momentum * self.velocity[key] - self.learning_rate * grads[key]
            # Update params
            updated_params[key] = params[key] + self.velocity[key]
        return updated_params

class RMSprop(Optimizer):
    # RMSprop: adapts lr per parameter
    def __init__(self, learning_rate=0.001, decay_rate=0.9, epsilon=1e-8):
        super().__init__(learning_rate)
        self.decay_rate = decay_rate
        self.epsilon = epsilon
        self.cache = {}  # Moving average of squared grads
    
    def update(self, params, grads):
        updated_params = {}
        for key in params:
            if key not in grads:
                raise ValueError(f"Gradient for parameter '{key}' not found")
            if key not in self.cache:
                self.cache[key] = np.zeros_like(params[key])
            # Update cache
            self.cache[key] = self.decay_rate * self.cache[key] + (1 - self.decay_rate) * grads[key] ** 2
            # Update params
            updated_params[key] = params[key] - self.learning_rate * grads[key] / (np.sqrt(self.cache[key]) + self.epsilon)
        return updated_params

class Adam(Optimizer):
    # Adam: combines Momentum and RMSprop
    def __init__(self, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        super().__init__(learning_rate)
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = {}  # First moment (momentum)
        self.v = {}  # Second moment (RMSprop)
        self.t = 0   # Time step for bias corection
    
    def update(self, params, grads):
        # Adam update with bias corection
        self.t += 1
        updated_params = {}
        for key in params:
            if key not in grads:
                raise ValueError(f"Gradient for parameter '{key}' not found")
            if key not in self.m:
                self.m[key] = np.zeros_like(params[key])
            if key not in self.v:
                self.v[key] = np.zeros_like(params[key])
            # Update moments
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * grads[key]
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * grads[key] ** 2
            # Bias corection
            m_hat = self.m[key] / (1 - self.beta1 ** self.t)
            v_hat = self.v[key] / (1 - self.beta2 ** self.t)
            # Update params
            updated_params[key] = params[key] - self.learning_rate * m_hat / (np.sqrt(v_hat) + self.epsilon)
        return updated_params

OPTIMIZERS = {
    'sgd': SGD,
    'momentum': MomentumSGD,
    'rmsprop': RMSprop,
    'adam': Adam
}

def get_optimizer(name, **kwargs):
    if name not in OPTIMIZERS:
        raise ValueError(f"Unknown optimizer: {name}")
    return OPTIMIZERS[name](**kwargs)


## 7. Neural Network

The floowing network class stacks layers, manage forward/backward passes, and compute loss and metrics.


Here, the `width` of a neural network corresponds to the number of hidden units in each layer, while its `depth` reflects the number of hidden layers. The total number of hidden units indicates the overall capacity of the model.


Letting $K$ denote the total number of layers, and $D_1, D_2, \dots, D_K$ is the number of hidden units in each layer. These are `hyperparameters`, meaning they are set before estimating the model’s parameters (such as weights and biases). With fixed hyperparameters, the model defines a class of functions, and the learned parameters determine the specific function selected from this class.

A deep neural network with multiple layers can be described using the following notation. Let $\mathbf{h}_k$ denote the vector of hidden units at layer $k$, $\boldsymbol{\beta}_k$ the bias vector for the next layer, and $\boldsymbol{\Omega}_k$ the weight matrix connecting layer $k$ to layer $k+1$. The network maps an input $\mathbf{x}$ to an output $\mathbf{y}$ through successive transformations:

$$
\begin{aligned}
\mathbf{h}_1 &= a(\boldsymbol{\beta}_0 + \boldsymbol{\Omega}_0 \mathbf{x}) \\
\mathbf{h}_2 &= a(\boldsymbol{\beta}_1 + \boldsymbol{\Omega}_1 \mathbf{h}_1) \\
&\;\;\vdots \\
\mathbf{h}_K &= a(\boldsymbol{\beta}_{K-1} + \boldsymbol{\Omega}_{K-1} \mathbf{h}_{K-1}) \\
\end{aligned}
$$


where $a[\cdot]$ is the activation function applied at each layer.\\


The output then becomes:

$$
\begin{aligned}
\mathbf{y} &= \boldsymbol{\beta}_K + \boldsymbol{\Omega}_K \mathbf{h}_K
\end{aligned}
$$

or as one single function:  

$$
y = \boldsymbol{\beta}_K + \boldsymbol{\Omega}_K a \boldsymbol{\beta}_{K-1} + \boldsymbol{\Omega}_{K-1} a \Big[ \dots \boldsymbol{\beta}_1 + \boldsymbol{\Omega}_1 a (\boldsymbol{\beta}_0 + \boldsymbol{\Omega}_0 \mathbf{x}) \dots \Big]
$$





In [ ]:
class NeuralNetwork:
    def __init__(self, input_size, hidden_layers, output_size, activation='relu',
                 output_activation='softmax', learning_rate=0.001, optimizer='adam',
                 weight_init='he', l2_lambda=0.0, dropout_rate=0.0, random_seed=None):
        # Initilize network layers
        self.layers = []
        input_dim = input_size
        # Create hidden layers
        for hidden_units in hidden_layers:
            layer = DenseLayer(input_size=input_dim, output_size=hidden_units,
                             activation=activation, weight_init=weight_init, seed=random_seed)
            self.layers.append(layer)
            input_dim = hidden_units
        # Output layer
        self.layers.append(DenseLayer(input_size=input_dim, output_size=output_size,
                                    activation=output_activation, weight_init=weight_init, seed=random_seed))
        # Store hyperparams
        self.input_size = input_size
        self.hidden_layers = hidden_layers
        self.output_size = output_size
        self.activation = activation
        self.output_activation = output_activation
        self.learning_rate = learning_rate
        self.optimizer_name = optimizer
        self.l2_lambda = l2_lambda
        self.dropout_rate = dropout_rate
        self.training = True
        # Setup optimzer
        self.optimizer = get_optimizer(optimizer, learning_rate=learning_rate)
        self.loss_function = 'cross_entropy'
        self.last_predictions = None
        self.last_loss = None
        self.dropout_masks = []
    
    def forward(self, X):
        # Forward pass with dropout during training
        A = X
        self.dropout_masks = []
        for i, layer in enumerate(self.layers):
            layer.activation_cache['A_prev'] = A
            # Linear transform
            Z = A @ layer.W + layer.b
            layer.activation_cache['Z'] = Z
            act = get_activation(layer.activation)
            A = act(Z)
            # Apply dropout to hidden layers
            is_hidden_layer = (i < len(self.layers) - 1)
            if self.training and is_hidden_layer and self.dropout_rate > 0.0:
                dropout_mask = (np.random.random(A.shape) > self.dropout_rate).astype(float)
                dropout_mask /= (1.0 - self.dropout_rate)
                A = A * dropout_mask
                self.dropout_masks.append(dropout_mask)
            else:
                self.dropout_masks.append(None)
            layer.activation_cache['A'] = A
        return A
    
    def backward(self, X, y, y_pred=None):
        # Backward pass: compute gradients
        if y_pred is None:
            y_pred = self.forward(X)
        loss_deriv_fn = get_loss_derivative(self.loss_function)
        dA = loss_deriv_fn(y_pred, y)
        m = X.shape[0]
        n_layers = len(self.layers)
        for idx in range(n_layers - 1, -1, -1):
            layer = self.layers[idx]
            Z = layer.activation_cache['Z']
            A_prev = layer.activation_cache['A_prev']
            # Skip softmax deriv if using softmax+crossentropy
            is_output = (idx == n_layers - 1)
            if is_output and self.output_activation == 'softmax' and self.loss_function == 'cross_entropy':
                dZ = dA
            else:
                activation_grad = get_activation_derivative(layer.activation)
                dZ = dA * activation_grad(Z)
            # Compute gradients (already averaged by loss deriv)
            layer.dW = A_prev.T @ dZ
            layer.db = np.sum(dZ, axis=0, keepdims=True)
            if layer.b.shape != layer.db.shape:
                layer.db = layer.db.reshape(layer.b.shape)
            # L2 regulariation
            if self.l2_lambda > 0:
                layer.dW += (self.l2_lambda / m) * layer.W
            # Propagate to prev layer
            dA = dZ @ layer.W.T
            # Apply dropout mask to gradient flowing into previous layer
            # The mask should match the one applied in forward pass for that layer
            if idx > 0:  # !!!! Not the first layer (input layer has no dropout)
                prev_layer_idx = idx - 1
                if prev_layer_idx < len(self.dropout_masks):
                    dropout_mask = self.dropout_masks[prev_layer_idx]
                    if dropout_mask is not None:
                        dA = dA * dropout_mask
    
    def update_weights(self):
        params = {}
        grads = {}
        for i, layer in enumerate(self.layers):
            params[f'W{i+1}'] = layer.W
            params[f'b{i+1}'] = layer.b
            grads[f'W{i+1}'] = layer.dW
            grads[f'b{i+1}'] = layer.db
        updated_params = self.optimizer.update(params, grads)
        for i, layer in enumerate(self.layers):
            layer.W = updated_params[f'W{i+1}']
            layer.b = updated_params[f'b{i+1}']
    
    def compute_loss(self, y_pred, y_true):
        loss_func = get_loss_function(self.loss_function)
        data_loss = loss_func(y_pred, y_true)
        if self.l2_lambda > 0:
            weights = [layer.W for layer in self.layers]
            reg_loss = l2_regularization(weights, self.l2_lambda)
            total_loss = data_loss + reg_loss
        else:
            total_loss = data_loss
        return total_loss
    
    def train_step(self, X_batch, y_batch):
        # One training step: forward -> backward -> update
        y_pred = self.forward(X_batch)
        loss = self.compute_loss(y_pred, y_batch)
        self.backward(X_batch, y_batch, y_pred=y_pred)
        self.update_weights()
        self.last_predictions = y_pred
        self.last_loss = loss
        return loss
    
    def predict(self, X):
        # Predict labels
        probabilities = self.predict_proba(X)
        predictions = np.argmax(probabilities, axis=1)
        return predictions
    
    def predict_proba(self, X):
        # Get prediction probabilites (no dropout)
        was_training = self.training
        self.training = False
        probabilities = self.forward(X)
        self.training = was_training
        return probabilities
    
    def train(self):
        self.training = True
    
    def eval(self):
        self.training = False
    
    def get_params(self):
        # Get all params
        params = {}
        for i, layer in enumerate(self.layers):
            params[f'W{i+1}'] = layer.W.copy()
            params[f'b{i+1}'] = layer.b.copy()
        return params
    
    def set_params(self, params):
        # Set params from dict
        for i, layer in enumerate(self.layers):
            layer.W = params[f'W{i+1}'].copy()
            layer.b = params[f'b{i+1}'].copy()



## 8. Training & Evaluation

[Back to top](#top)

During training, **Dropout** and **Early stopping** has furthermore been included.  

#### Early stopping

Early stopping leverages the validation set to detect overfitting during training. The validation objective is monitored across epochs, and training is halted once this objective begins to increase. This prevents the model from fitting noise in the training data and functions as an effective regularization mechanism.

#### Dropout 

Dropout is a regularization technique to prevent overfitting. During training, a random fraction of neurons is deactivated at each step, so the number of active nodes and their connectivity vary over time. Each weight update is computed with a different subset of the network, as training multiple networks in parallel. Dropout reduces overfitting and improves generalization. During inference or testing, dropout is disabled and all neurons remain active.